# 01 · Kohn-Sham DFT: grids, numerical integration, LDA/GGA/meta-GGA

这份 notebook 面向已经熟悉 RHF/FCI 的同学：我们不从交换相关泛函的历史讲起，而是从代码里真正发生的事情讲起。

本节目标：

1. 看清 Kohn-Sham DFT 和 RHF 的结构差别
2. 理解为什么 DFT 需要格点和数值积分
3. 区分 LDA、GGA、meta-GGA 依赖的局域变量
4. 手写一个纯泛函 RKS SCF 外层循环
5. 与 PySCF RKS 高精度对照

本系列只考虑纯泛函，不讨论 hybrid functional。

---

## 1 · Kohn-Sham 方程和 RHF 的相似处

RHF 的 Fock 矩阵是：

$$
F^{\text{RHF}} = h + J - \frac{1}{2}K
$$

纯泛函 Kohn-Sham DFT 的有效矩阵是：

$$
F^{\text{KS}} = h + J + V_{\text{xc}}[\rho]
$$

也就是说，对闭壳层纯泛函 RKS：

| 项 | RHF | RKS pure DFT |
|----|-----|--------------|
| core Hamiltonian | $h$ | $h$ |
| Coulomb | $J$ | $J$ |
| exchange | exact exchange $K$ | 没有显式 $K$ |
| exchange-correlation | 无 | $E_{\text{xc}}[\rho]$ 和 $V_{\text{xc}}$ |

总能量为：

$$
E_{\text{RKS}} = \mathrm{Tr}(Dh) + \frac{1}{2}\mathrm{Tr}(DJ) + E_{\text{xc}}[\rho] + E_{\text{nuc}}
$$

新的难点是 $E_{\text{xc}}[\rho]$ 通常没有简单的 AO 积分表达式，需要在 real-space grid 上数值积分。

---

## 2 · 分子、积分和初始 RHF 密度

In [1]:
import numpy as np
from scipy.linalg import eigh, fractional_matrix_power
from pyscf import gto, dft

# -------- H2O / STO-3G: 小体系便于看清数组 --------
mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="sto-3g")

nao = mol.nao_nr()
nelec = mol.nelectron
nocc = nelec // 2

S = mol.intor("int1e_ovlp_sph")
T = mol.intor("int1e_kin_sph")
V = mol.intor("int1e_nuc_sph")
H_core = T + V
eri = mol.intor("int2e_sph")
A = fractional_matrix_power(S, -0.5)
E_nuc = mol.energy_nuc()

print(f"nao = {nao}, nelec = {nelec}, nocc = {nocc}")
print(f"H_core shape = {H_core.shape}")
print(f"eri shape = {eri.shape}")
print(f"E_nuc = {E_nuc:.10f} Hartree")

nao = 7, nelec = 10, nocc = 5
H_core shape = (7, 7)
eri shape = (7, 7, 7, 7)
E_nuc = 9.1882584177 Hartree


---

## 3 · 格点与数值积分

DFT 的交换相关能写成格点求和：

$$
E_{\text{xc}} \approx \sum_g w_g\, \rho(\mathbf r_g)\, \varepsilon_{\text{xc}}(\mathbf r_g)
$$

PySCF 的 grid 包含：

- `coords[g]`: 第 $g$ 个格点的三维坐标
- `weights[g]`: 第 $g$ 个格点的积分权重

不同泛函需要不同局域变量：

| 类型 | 依赖变量 | 直觉 |
|------|----------|------|
| LDA | $\rho$ | 只看当前位置电子密度 |
| GGA | $\rho, \nabla\rho$ | 还看密度变化有多快 |
| meta-GGA | $\rho, \nabla\rho, \tau$ 等 | 再加入 kinetic-energy density 等半局域信息 |

这里的 $\tau$ 是 occupied KS orbital 的 kinetic-energy density，不是 correlated kinetic energy。

In [2]:
# 先用 core Hamiltonian 产生一个初始密度，用来演示 grid 上的 rho。
def get_occ(mo_energy):
    idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[idx[:nocc]] = 2.0
    return mo_occ


def make_rdm1(mo_coeff, mo_occ):
    C_occ = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (C_occ * occ) @ C_occ.T

mo_energy0, mo_coeff0 = eigh(H_core, S)
dm0 = make_rdm1(mo_coeff0, get_occ(mo_energy0))

# PySCF grid: level 越高，格点越密。
grids = dft.gen_grid.Grids(mol)
grids.level = 3
grids.build()

ni = dft.numint.NumInt()
coords = grids.coords
weights = grids.weights

print(f"number of grid points = {len(weights):,}")
print(f"coords shape = {coords.shape}")
print(f"weights shape = {weights.shape}")
print(f"sum(weights) is not a volume to interpret directly: {weights.sum():.6f}")

number of grid points = 33,704
coords shape = (33704, 3)
weights shape = (33704,)
sum(weights) is not a volume to interpret directly: 16836.577611


In [3]:
# LDA: rho(g)
ao_lda = ni.eval_ao(mol, coords, deriv=0)
rho_lda = ni.eval_rho(mol, ao_lda, dm0, xctype="LDA")

# GGA: rho, d rho/dx, d rho/dy, d rho/dz
ao_gga = ni.eval_ao(mol, coords, deriv=1)
rho_gga = ni.eval_rho(mol, ao_gga, dm0, xctype="GGA")

# meta-GGA: rho, gradient, laplacian, tau in PySCF's convention
ao_mgga = ni.eval_ao(mol, coords, deriv=2)
rho_mgga = ni.eval_rho(mol, ao_mgga, dm0, xctype="MGGA")

print(f"ao_lda shape  = {ao_lda.shape}       # (ngrid, nao)")
print(f"rho_lda shape = {rho_lda.shape}       # rho")
print(f"rho_gga shape = {rho_gga.shape}       # rho + 3 gradient components")
print(f"rho_mgga shape = {rho_mgga.shape}     # rho + grad + laplacian + tau")

nelec_grid = np.dot(weights, rho_lda)
print(f"integral rho(r) dr = {nelec_grid:.10f} electrons")

ao_lda shape  = (33704, 7)       # (ngrid, nao)
rho_lda shape = (33704,)       # rho
rho_gga shape = (4, 33704)       # rho + 3 gradient components
rho_mgga shape = (6, 33704)     # rho + grad + laplacian + tau
integral rho(r) dr = 10.0000001128 electrons


---

## 4 · 用 PySCF 的 numint 得到 $E_{xc}$ 和 $V_{xc}$

在教学代码里，我们不手写 LDA/PBE/TPSS 的解析公式。PySCF 的 `numint.nr_rks` 会做三件事：

1. 在每个 grid 上计算 $\rho$、$\nabla\rho$、$\tau$ 等变量
2. 调用 libxc / xcfun 得到 $\varepsilon_{xc}$ 和 functional derivative
3. 把 $V_{xc}(\mathbf r)$ 积回 AO 矩阵 $V_{xc,\mu\nu}$

返回值是：

```python
nelec_grid, exc, vxc = ni.nr_rks(mol, grids, xc, dm)
```

其中 `exc` 是已经积分完的 $E_{xc}$，`vxc` 是 AO 表象的矩阵。

In [4]:
for name, xc in [("LDA", "lda,vwn"), ("GGA", "pbe,pbe"), ("meta-GGA", "tpss,tpss")]:
    nelec_grid, exc, vxc = ni.nr_rks(mol, grids, xc, dm0)
    print(f"{name:8s}  xc={xc:9s}  nelec={nelec_grid:.8f}  Exc={exc: .10f}  vxc shape={vxc.shape}")

LDA       xc=lda,vwn    nelec=10.00000011  Exc=-9.7775350106  vxc shape=(7, 7)
GGA       xc=pbe,pbe    nelec=10.00000011  Exc=-10.2725835118  vxc shape=(7, 7)
meta-GGA  xc=tpss,tpss  nelec=10.00000011  Exc=-10.3800764759  vxc shape=(7, 7)


---

## 5 · RKS helper functions

RKS SCF 和 RHF 的外层几乎一样：给定密度 $D$，构造 $F$，对角化，再更新密度。

区别只在于：

$$
F = h + J + V_{xc}
$$

这里仍然用 RHF 章节的 DIIS 思路加速收敛。

In [5]:
def get_j(dm):
    return np.einsum("ijkl,kl->ij", eri, dm, optimize=True)


def get_err_vec(fock, dm):
    return A @ (fock @ dm @ S - S @ dm @ fock) @ A


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    B = np.empty((n + 1, n + 1))
    B[-1, :] = -1.0
    B[:, -1] = -1.0
    B[-1, -1] = 0.0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.einsum("ij,ij->", err_vec_list[i], err_vec_list[j], optimize=True)

    rhs = np.zeros(n + 1)
    rhs[-1] = -1.0
    coeff = np.linalg.solve(B, rhs)[:-1]
    return np.einsum("i,ijk->jk", coeff, np.array(fock_list), optimize=True)


def energy_rks(dm, J, exc):
    e1 = np.einsum("ij,ji->", H_core, dm, optimize=True)
    e_coul = 0.5 * np.einsum("ij,ji->", J, dm, optimize=True)
    return e1 + e_coul + exc + E_nuc

print("RKS helper functions ready.")

RKS helper functions ready.


---

## 6 · 完整 RKS + DIIS

In [6]:
def run_rks(xc, grid_level=3, conv_tol=1e-10, max_cycle=80, diis_space=8):
    grids = dft.gen_grid.Grids(mol)
    grids.level = grid_level
    grids.build()
    ni = dft.numint.NumInt()

    mo_energy, mo_coeff = eigh(H_core, S)
    mo_occ = get_occ(mo_energy)
    dm = make_rdm1(mo_coeff, mo_occ)

    e_old = 0.0
    fock_list = []
    err_vec_list = []

    for cycle in range(max_cycle):
        J = get_j(dm)
        nelec_grid, exc, vxc = ni.nr_rks(mol, grids, xc, dm)
        fock = H_core + J + vxc
        e_tot = energy_rks(dm, J, exc)

        fock_list.append(fock)
        err_vec_list.append(get_err_vec(fock, dm))
        if len(fock_list) > diis_space:
            fock_list.pop(0)
            err_vec_list.pop(0)

        fock_diis = apply_diis(fock_list, err_vec_list) if cycle >= 2 else fock

        mo_energy, mo_coeff = eigh(fock_diis, S)
        mo_occ = get_occ(mo_energy)
        dm_new = make_rdm1(mo_coeff, mo_occ)

        dE = abs(e_tot - e_old)
        dD = np.linalg.norm(dm_new - dm)
        if dE < conv_tol and dD < 1e-8:
            return {
                "xc": xc,
                "e_tot": e_tot,
                "exc": exc,
                "nelec_grid": nelec_grid,
                "cycles": cycle + 1,
                "mo_energy": mo_energy,
                "dm": dm,
                "ngrid": len(grids.weights),
            }

        dm = dm_new
        e_old = e_tot

    raise RuntimeError(f"RKS did not converge for xc={xc}")

print("run_rks is ready.")

run_rks is ready.


---

## 7 · LDA、GGA、meta-GGA 对照

使用三个纯泛函：

| 类型 | PySCF 写法 | 说明 |
|------|------------|------|
| LDA | `lda,vwn` | local density approximation |
| GGA | `pbe,pbe` | 使用 density gradient |
| meta-GGA | `tpss,tpss` | 使用 kinetic-energy density 等半局域变量 |

同一个 RKS 外层循环可以处理三者；差别藏在 `numint.nr_rks` 对局域变量和 functional derivative 的处理中。

In [7]:
functionals = [
    ("LDA", "lda,vwn"),
    ("GGA", "pbe,pbe"),
    ("meta-GGA", "tpss,tpss"),
]

print(f"{'type':10s} {'xc':10s} {'E_ours':>18s} {'E_PySCF':>18s} {'diff':>12s} {'cyc':>4s}")
print("-" * 78)

for kind, xc in functionals:
    ours = run_rks(xc, grid_level=3)

    mf_ref = dft.RKS(mol)
    mf_ref.xc = xc
    mf_ref.grids.level = 3
    mf_ref.conv_tol = 1e-10
    mf_ref.verbose = 0
    mf_ref.kernel()

    diff = ours["e_tot"] - mf_ref.e_tot
    print(f"{kind:10s} {xc:10s} {ours['e_tot']:18.10f} {mf_ref.e_tot:18.10f} {diff:12.2e} {ours['cycles']:4d}")

type       xc                     E_ours            E_PySCF         diff  cyc
------------------------------------------------------------------------------
LDA        lda,vwn        -74.7321051891     -74.7321051891     1.42e-14   10
GGA        pbe,pbe        -75.2256398123     -75.2256398123    -1.42e-14   10
meta-GGA   tpss,tpss      -75.3267977772     -75.3267977772    -5.68e-14    9


---

## 8 · 小结

| 概念 | 代码对应 |
|------|----------|
| grid points | `grids.coords` |
| quadrature weights | `grids.weights` |
| density on grid | `ni.eval_rho(...)` |
| LDA | 只依赖 $\rho(\mathbf r)$ |
| GGA | 依赖 $\rho(\mathbf r)$ 和 $\nabla\rho(\mathbf r)$ |
| meta-GGA | 再依赖 $\tau(\mathbf r)$ 等半局域变量 |
| pure RKS Fock | `H_core + J + vxc` |
| XC 数值积分 | `ni.nr_rks(mol, grids, xc, dm)` |

对 WFT 背景的同学来说，可以把纯泛函 DFT 理解成：保留 RHF 的 SCF 框架和 Coulomb $J$，把 exact exchange/correlation 换成一个在实空间格点上积分得到的 $E_{xc}[\rho]$ 和 $V_{xc}$。